# Project Milestone
## Skin-Lesion Classification on HAM10000 — Progress Report

**Course:** Introduction to Data Science
**Instructor:** Quách Đình Hoàng
**Team 07:** Nguyễn Nhật Phát, Bùi Trần Tấn Phát
**Milestone deadline:** 12 April 2026

---

### Table of contents
1. [Member contributions so far](#1)
2. [Project recap](#2)
3. [Work completed](#3)
4. [Preliminary results](#4)
5. [Discussion of preliminary results](#5)
6. [Plan for the remaining weeks](#6)

## 1. Member contributions so far  <a id="1"></a>

| Member | What they did (up to milestone) | Approx. share |
|---|---|---|
| **Nguyễn Nhật Phát** | Dataset acquisition and description; EDA on metadata; Chi-square tests (`sex`/`dx`, `localization`/`dx`); Kruskal–Wallis on age; Dunn post-hoc; figure generation for distributions; drafting of proposal and milestone narrative. | 50% |
| **Bùi Trần Tấn Phát** | Handcrafted image-feature pipeline (RGB/HSV, brightness, contrast, asymmetry, border irregularity, LBP, GLCM — 50 descriptors); preprocessing (one-hot, SMOTE, StandardScaler); baseline training of LR, KNN, RF, SVM, XGB, LightGBM with stratified 5-fold CV; feature-set comparison experiment. | 50% |
| *Shared* | Weekly code review, results discussion, Git workflow, milestone export. | — |

Both members are on track with the plan submitted in the Proposal.

## 2. Project recap  <a id="2"></a>

**Problem.** Multi-class classification of dermatoscopic lesions from the HAM10000 dataset into seven diagnostic categories, with particular emphasis on recall for the melanoma (`mel`) class because of its clinical weight.

**Dataset.** 10,015 dermatoscopic images with metadata (`lesion_id`, `image_id`, `dx`, `dx_type`, `age`, `sex`, `localization`). Class distribution is strongly skewed toward melanocytic nevi (≈ 67%).

**Inputs / outputs.**
- Y: `dx` (7-class).
- X: 4 metadata predictors + 50 handcrafted image features → 50 input features after one-hot encoding.

**Evaluation.** Primary — **Macro-F1** and **melanoma recall**. Secondary — Accuracy, Weighted-F1, OvR ROC-AUC, confusion matrices.

## 3. Work completed  <a id="3"></a>

- [x] Data audit: no duplicate `image_id`; ~0.6% missing `age` → median imputation.
- [x] Class-distribution EDA (see figure below).
- [x] Chi-square tests:
  - `sex` vs `dx` → statistically significant (p < 0.01).
  - `localization` vs `dx` → statistically significant (p < 0.001).
- [x] Kruskal–Wallis on `age` across `dx` → reject H0 (p < 0.001).
- [x] Dunn post-hoc with Bonferroni correction → located pairwise age differences (notably `nv` vs `mel`, `nv` vs `bcc`).
- [x] Handcrafted image-feature extraction (50 descriptors) cached to `results/image_features.csv`.
- [x] Baseline models trained with 5-fold Stratified CV: Logistic Regression, KNN, Random Forest, SVM (RBF), XGBoost, LightGBM.
- [x] Feature-set ablation: Metadata-only vs Image-only vs Combined.
- [ ] *Pending:* full hyperparameter search on LightGBM; melanoma-focused two-tier pipeline; final presentation slides.

## 4. Preliminary results  <a id="4"></a>

The numbers below are loaded directly from the cached experiment output `../results/results_summary.json` so that this notebook stays consistent with the Report.

In [ ]:
import json
from pathlib import Path
import pandas as pd

RESULTS_PATH = Path('..') / 'results' / 'results_summary.json'
with RESULTS_PATH.open('r', encoding='utf-8') as f:
    results = json.load(f)

pd.set_option('display.float_format', lambda x: f'{x:.4f}')

### 4.1 Baseline model comparison (all features, default hyperparameters)

In [ ]:
model_cmp = pd.DataFrame(results['model_comparison']).sort_values('Macro F1', ascending=False).reset_index(drop=True)
model_cmp

**Reading.** Gradient-boosted trees lead on both Macro-F1 and accuracy: XGBoost reaches **0.815 accuracy / 0.578 macro-F1**, LightGBM **0.817 / 0.575**. Random Forest and Logistic Regression form a second tier; KNN lags far behind, which is expected in a 50-dim feature space dominated by categorical one-hot columns. We will therefore focus the hyperparameter-tuning effort on LightGBM/XGBoost for the final report.

In [ ]:
from IPython.display import Image
Image(filename=str(Path('..') / 'results' / 'model_comparison.png'))

### 4.2 Feature-set ablation (metadata vs image vs combined)

In [ ]:
fs = pd.DataFrame(results['feature_set_comparison']).T
fs.index.name = 'Feature set'
fs.reset_index()

**Reading.** The combined feature set (metadata + image descriptors, 50 features) is clearly the strongest, beating metadata-only by **+0.097 accuracy** and image-only by **+0.056 accuracy**. ROC-AUC climbs from 0.875 (metadata only) to **0.946** (combined). This directly addresses **RQ3** in the Proposal: the two feature groups are complementary, not redundant.

In [ ]:
Image(filename=str(Path('..') / 'results' / 'feature_set_comparison.png'))

### 4.3 Best model so far — Tuned LightGBM

In [ ]:
bm = results['best_model']
pd.Series({
    'Model':       bm['name'],
    'Accuracy':    bm['accuracy'],
    'Macro F1':    bm['macro_f1'],
    'Weighted F1': bm['weighted_f1'],
    'ROC-AUC':     bm['roc_auc'],
}).to_frame('Value')

In [ ]:
pd.Series(bm['best_params']).to_frame('Best LightGBM hyperparameter')

### 4.4 Melanoma-specific performance (baseline)

In [ ]:
mm = results['melanoma_metrics']
pd.Series({'Recall': mm['recall'], 'Precision': mm['precision'], 'F1': mm['f1']}).to_frame('Melanoma (baseline)')

**Reading.** At the default argmax decision rule, melanoma recall is only **0.54** — meaning the model misses almost half of actual melanoma cases. This is the single biggest clinical weakness of the baseline and is the focus of the remaining work.

### 4.5 Supporting figures

In [ ]:
Image(filename=str(Path('..') / 'results' / 'class_distribution.png'))

In [ ]:
Image(filename=str(Path('..') / 'results' / 'confusion_matrices.png'))

In [ ]:
Image(filename=str(Path('..') / 'results' / 'roc_curves.png'))

## 5. Discussion of preliminary results  <a id="5"></a>

1. **Statistical tests (RQ1) are conclusive.** Both `sex` and `localization` are significantly associated with `dx`, and age distributions differ between diagnostic groups. This confirms that metadata carries useful predictive signal and is not just a nuisance covariate.
2. **Combined features dominate (RQ3).** The feature-set ablation shows a clear ordering `Metadata < Image < Combined` on every metric we care about. This is an important finding for the final report because it motivates the whole pipeline design.
3. **The overall classifier is competitive but melanoma recall is not.** A Macro-F1 of ~0.58 and ROC-AUC ≈ 0.95 are solid for tabular ML on HAM10000, but a 0.54 recall on `mel` is clinically unacceptable. Most errors are `mel → nv` confusions, consistent with the dermatological literature.
4. **Class imbalance is the root cause.** Even with `class_weight='balanced'` and SMOTE, the dominant `nv` class biases the default argmax decision. This tells us that **threshold tuning** and/or a **dedicated binary mel-vs-rest classifier** are the right next steps, rather than chasing yet another model family.

## 6. Plan for the remaining weeks  <a id="6"></a>

| Week | Action | Expected deliverable |
|---|---|---|
| 12 – 18 Apr | Randomised hyperparameter search on LightGBM with 5-fold CV; refresh feature-importance analysis. | Updated `results_summary.json`, feature-importance plots. |
| 12 – 18 Apr | Train a **binary mel-vs-rest LightGBM** with aggressive SMOTE and `scale_pos_weight`; sweep decision threshold. | `mel_improvement_summary.json`, ROC/PR curves for mel. |
| 19 – 22 Apr | **Two-tier pipeline**: binary gate + 7-class model; threshold sweep on the 7-class mel probability. | Side-by-side table of (baseline, threshold-tuned, binary, two-tier). |
| 19 – 22 Apr | Presentation rehearsal, slide polish, 10-minute timing. | `Presentation/Presentation.pptx`. |
| 23 – 26 Apr | Final Report narrative, reproducibility appendix, HTML/PDF export, peer assessment. | `Report/Report.ipynb` + `Report.html`. |

**Risks & mitigations.**
- *Image-feature extraction is slow (~10 min).* → Cache the output in `results/image_features.csv` so modelling experiments become cheap and reproducible.
- *Hyperparameter search may over-fit on the validation folds.* → Report test-set metrics only once after model selection; never use the test split for tuning.
- *Two-tier pipeline could hurt macro-F1.* → Always report *both* the melanoma recall and the overall macro-F1 so the trade-off is explicit.